<a href="https://colab.research.google.com/github/bashoori/2015lab4/blob/master/clinical_nlp_ensemble_pathology.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [27]:
%cd /content
!git clone https://github.com/bashoori/clinical-nlp-ensemble.git



/content
fatal: destination path 'clinical-nlp-ensemble' already exists and is not an empty directory.


In [14]:
from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
import pandas as pd
import numpy as np

SEARCH_ROOT = Path("/content/drive/MyDrive/Colab Notebooks/UBC-Extraction-and-Analysis-of-Clinical-Documents")
FILE_NAME = "synthetic_breast_cancer_reports_600.csv"
matches = list(SEARCH_ROOT.rglob(FILE_NAME))
if not matches:
    raise FileNotFoundError(f"Could not find {FILE_NAME} under {SEARCH_ROOT}. Update SEARCH_ROOT if needed.")
FILE_PATH = matches[0]
print("Using dataset:", FILE_PATH)

df = pd.read_csv(FILE_PATH)
print("Rows:", len(df))
print("Columns:", df.columns.tolist())
display(df.head())

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using dataset: /content/drive/MyDrive/Colab Notebooks/UBC-Extraction-and-Analysis-of-Clinical-Documents/synthetic_breast_cancer_reports_600.csv
Rows: 600
Columns: ['Unnamed: 0', 'report', 'lvi_label']


,Unnamed: 0,report,lvi_label
0,0,Sentinel nodes negative.\nNeural involvement: ...,0
1,1,Sentinel lymph node: 1/3 positive.\nMargins in...,1
2,2,"Diagnosis: Invasive ductal carcinoma, Grade II...",0
3,3,"Tumor size: 2.4 cm.\nHormone receptors: ER +, ...",1
4,4,"ER: Positive, PR: Positive, HER2: Negative.\nD...",0


# Clinical NLP Ensemble for Pathology Reports
**UBC Module 2 learning project — synthetic/course-assignment data**

This notebook explores LVI extraction from pathology-report text, compares model approaches, combines predictions, and flags uncertain cases for human review.

**Scope:** The supplied screenshot shows columns `report` and `lvi_label`, so this notebook evaluates LVI. It does not train a Nottingham-grade classifier because no Nottingham-grade target column is shown. This is an educational prototype, not a clinical decision system.

## 1. Load the CSV from Google Drive

The screenshot shows `synthetic_breast_cancer_reports_600.csv` under `MyDrive/Colab Notebooks/...`, but the full folder path is cropped. This cell searches that folder for the exact filename.

## 2. Check and prepare the data

Expected columns: `report` (text) and `lvi_label` (`1` = LVI present, `0` = LVI absent). An exported index column is not a clinical feature and will be dropped if present.

In [15]:
required = {"report", "lvi_label"}
missing = required - set(df.columns)
if missing:
    raise ValueError(f"Missing required columns: {missing}; found {df.columns.tolist()}")

df = df.drop(columns=["Unnamed: 0"], errors="ignore").copy()
df["report"] = df["report"].fillna("").astype(str)
df["lvi_label"] = pd.to_numeric(df["lvi_label"], errors="coerce")
df = df.dropna(subset=["lvi_label"]).copy()
df["lvi_label"] = df["lvi_label"].astype(int)
if not set(df["lvi_label"].unique()).issubset({0, 1}):
    raise ValueError("Expected binary lvi_label values 0 and 1.")

print("Shape:", df.shape)
display(df["lvi_label"].value_counts().sort_index().rename(index={0:"LVI absent (0)", 1:"LVI present (1)"}))
print("Empty reports:", (df["report"].str.strip() == "").sum())
display(df[["report", "lvi_label"]].head(10))

Shape: (600, 2)


,count
lvi_label,
LVI absent (0),322
LVI present (1),278


Empty reports: 0


,report,lvi_label
0,Sentinel nodes negative.\nNeural involvement: ...,0
1,Sentinel lymph node: 1/3 positive.\nMargins in...,1
2,"Diagnosis: Invasive ductal carcinoma, Grade II...",0
3,"Tumor size: 2.4 cm.\nHormone receptors: ER +, ...",1
4,"ER: Positive, PR: Positive, HER2: Negative.\nD...",0
5,"Tumor size: 2.4 cm.\nSize: 1.8 cm, Grade III I...",0
6,Sentinel lymph node: 1/3 positive.\nDiagnosis:...,1
7,"Diagnosis: Invasive ductal carcinoma, Grade II...",0
8,Sentinel lymph node: 1/3 positive.\nSentinel n...,1
9,"Tumor size: 2.4 cm.\nSize: 1.8 cm, Grade III I...",0


## 3. Train/test split
Hold out reports for evaluation so we measure performance on data not used for training.

In [16]:
from sklearn.model_selection import train_test_split
X, y = df["report"], df["lvi_label"]
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.20, random_state=42, stratify=y
)
print("Train:", len(X_train), "| Test:", len(X_test))

Train: 480 | Test: 120


## 4. Model A — rule-based baseline

This baseline searches for common positive and negative phrases. It abstains if it finds both polarities or neither, rather than pretending to know the answer.

In [17]:
import re

NEGATIVE_PATTERNS = [
    r"no lymphovascular invasion",
    r"lymphovascular invasion is not identified",
    r"lymphovascular invasion not identified",
    r"negative for lymphovascular invasion",
    r"no evidence of lymphovascular invasion",
    r"lvi is absent",
    r"lvi negative",
]
POSITIVE_PATTERNS = [
    r"lymphovascular invasion is present",
    r"lymphovascular invasion present",
    r"positive for lymphovascular invasion",
    r"identified lymphovascular invasion",
    r"lvi is present",
    r"lvi positive",
]

def rule_based_predict(text):
    text = str(text).lower()
    neg = any(re.search(p, text) for p in NEGATIVE_PATTERNS)
    pos = any(re.search(p, text) for p in POSITIVE_PATTERNS)
    if neg and not pos: return 0
    if pos and not neg: return 1
    return np.nan

rule_test = X_test.apply(rule_based_predict)
print("Rule-based coverage:", round(rule_test.notna().mean(), 3))
print("Abstained/uncertain:", int(rule_test.isna().sum()))

Rule-based coverage: 0.133
Abstained/uncertain: 104


## 5. Model B — classical machine learning

TF-IDF converts text into features and Logistic Regression learns patterns from training reports. This model provides class scores used for the probability-based example.

In [5]:
from sklearn.pipeline import Pipeline
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

ml_model = Pipeline([
    ("tfidf", TfidfVectorizer(ngram_range=(1, 2), max_features=20000)),
    ("classifier", LogisticRegression(max_iter=1000, class_weight="balanced", random_state=42)),
])
ml_model.fit(X_train, y_train)
ml_pred = ml_model.predict(X_test)
ml_prob = ml_model.predict_proba(X_test)[:, 1]
print("Classical ML model trained.")

Classical ML model trained.


## 6. Model C — optional fine-tuned transformer

This section is optional because transformer training may take longer and use more compute. If enabled, it fine-tunes a general-purpose transformer on this training split; it is not a medically validated model.

In [6]:
RUN_TRANSFORMER = False  # Change to True to install dependencies and fine-tune.

if RUN_TRANSFORMER:
    !pip -q install transformers datasets accelerate
    import torch
    from datasets import Dataset
    from transformers import AutoTokenizer, AutoModelForSequenceClassification, TrainingArguments, Trainer
    from sklearn.metrics import accuracy_score, f1_score

    MODEL_NAME = "distilbert-base-uncased"
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
    train_ds = Dataset.from_dict({"text": X_train.tolist(), "label": y_train.tolist()})
    test_ds = Dataset.from_dict({"text": X_test.tolist(), "label": y_test.tolist()})

    def tokenize(batch):
        return tokenizer(batch["text"], truncation=True, padding="max_length", max_length=256)

    train_ds = train_ds.map(tokenize, batched=True).remove_columns(["text"])
    test_ds = test_ds.map(tokenize, batched=True).remove_columns(["text"])
    train_ds.set_format("torch")
    test_ds.set_format("torch")
    transformer = AutoModelForSequenceClassification.from_pretrained(MODEL_NAME, num_labels=2)

    def compute_metrics(eval_pred):
        logits, labels = eval_pred
        preds = np.argmax(logits, axis=1)
        return {"accuracy": accuracy_score(labels, preds), "f1": f1_score(labels, preds, zero_division=0)}

    args = TrainingArguments(
        output_dir="./transformer_lvi_model", num_train_epochs=2,
        per_device_train_batch_size=8, per_device_eval_batch_size=8,
        save_strategy="no", logging_steps=20, report_to="none", seed=42
    )
    trainer = Trainer(model=transformer, args=args, train_dataset=train_ds,
                      eval_dataset=test_ds, compute_metrics=compute_metrics)
    trainer.train()
    output = trainer.predict(test_ds)
    transformer_prob = torch.softmax(torch.tensor(output.predictions), dim=1)[:, 1].numpy()
    transformer_pred = (transformer_prob >= 0.5).astype(int)
else:
    print("Transformer skipped. Set RUN_TRANSFORMER=True to run it.")

Transformer skipped. Set RUN_TRANSFORMER=True to run it.


## 7. Model D — optional LLM

A real LLM requires a configured provider/API and may incur cost. No fake LLM predictions are generated here. Add an approved provider integration only if needed; never publish API keys in a GitHub notebook.

In [7]:
USE_REAL_LLM = False
if USE_REAL_LLM:
    raise NotImplementedError("Add your approved LLM provider call here; keep API keys in Colab Secrets.")
else:
    print("No external LLM was called. This run uses the rule-based and classical ML approaches.")

No external LLM was called. This run uses the rule-based and classical ML approaches.


## 8. Combine predictions

Methods:
- **Majority voting:** choose the class supported by the most model votes.
- **Probability averaging:** average probabilities, then apply a threshold.
- **Weighted confidence:** give more influence to models based on validation performance.

The rule-based model provides labels or abstains, not calibrated probabilities. Therefore, this starter version uses the classical ML model's probabilities for probability averaging. Weighted confidence requires a validation split to estimate weights and is left as a transparent extension rather than inventing weights.

In [8]:
# Majority vote from the rule-based model and classical ML.
# If the rule-based model abstains or there is a tie, use the ML prediction as the documented fallback.
majority_predictions = []
for rule_value, ml_value in zip(rule_test.to_numpy(), ml_pred):
    votes = [] if pd.isna(rule_value) else [int(rule_value)]
    votes.append(int(ml_value))
    counts = pd.Series(votes).value_counts()
    if len(counts) > 1 and counts.iloc[0] == counts.iloc[1]:
        majority_predictions.append(int(ml_value))
    else:
        majority_predictions.append(int(counts.index[0]))
majority_predictions = np.array(majority_predictions)

# Probability averaging example from the lecture:
example_probabilities = [0.45, 0.45, 0.90]
example_average = float(np.mean(example_probabilities))
print("Lecture probabilities:", example_probabilities)
print("Average:", round(example_average, 2))
print("Classification at threshold 0.5:", "LVI present" if example_average >= 0.5 else "LVI absent")

# Current data-driven probability prediction from the trained ML model.
probability_average_predictions = (ml_prob >= 0.5).astype(int)

Lecture probabilities: [0.45, 0.45, 0.9]
Average: 0.6
Classification at threshold 0.5: LVI present


## 9. Flag cases for human review

Flag reports when the rule-based model abstains, when rule-based and ML predictions disagree, or when the ML score is close to 0.5. Thresholds in a real clinical system must be justified and validated by domain experts.

In [9]:
review_flags, review_reasons = [], []
for rule_value, ml_value, prob in zip(rule_test.to_numpy(), ml_pred, ml_prob):
    reasons = []
    if pd.isna(rule_value):
        reasons.append("Rule-based model uncertain/conflicting")
    elif int(rule_value) != int(ml_value):
        reasons.append("Models disagree")
    if abs(float(prob) - 0.5) <= 0.10:
        reasons.append("ML score near threshold")
    review_flags.append(bool(reasons))
    review_reasons.append("; ".join(reasons))

results = pd.DataFrame({
    "report": X_test.reset_index(drop=True),
    "actual_lvi_label": y_test.reset_index(drop=True),
    "rule_prediction": rule_test.reset_index(drop=True),
    "ml_prediction": ml_pred,
    "ml_score_lvi_present": ml_prob,
    "majority_vote_prediction": majority_predictions,
    "human_review": review_flags,
    "review_reason": review_reasons,
})
display(results.head(10))

,report,actual_lvi_label,rule_prediction,ml_prediction,ml_score_lvi_present,majority_vote_prediction,human_review,review_reason
0,Neural involvement: None.\nSentinel lymph node...,0,NaN,0,0.233401,0,True,Rule-based model uncertain/conflicting
1,LVI noted. Sentinel lymph node biopsy negative...,1,NaN,1,0.764762,1,True,Rule-based model uncertain/conflicting
2,Neural involvement: None.\nDiagnosis: Invasive...,1,NaN,1,0.575152,1,True,Rule-based model uncertain/conflicting; ML sco...
3,Evidence of lymphovascular invasion. No DCIS i...,1,NaN,1,0.544390,1,True,Rule-based model uncertain/conflicting; ML sco...
4,"Diagnosis: Invasive ductal carcinoma, Grade II...",0,NaN,0,0.345160,0,True,Rule-based model uncertain/conflicting
5,LVI not seen. Perineural invasion: absent. His...,0,NaN,0,0.180351,0,True,Rule-based model uncertain/conflicting
6,Margins involved by tumor.\nPerineural invasio...,1,NaN,1,0.773590,1,True,Rule-based model uncertain/conflicting
7,Evidence of lymphovascular invasion. HER2: equ...,1,NaN,1,0.532677,1,True,Rule-based model uncertain/conflicting; ML sco...
8,Sentinel lymph node: 1/3 positive.\nHormone re...,1,1.0,1,0.765933,1,False,
9,Extensive necrosis present.\nNo necrosis obser...,1,NaN,0,0.495434,0,True,Rule-based model uncertain/conflicting; ML sco...


## 10. Evaluate performance
Compare the classical ML model, the combined rule-plus-ML decision, and rule-based coverage. Metrics on synthetic/course-assignment data are for learning, not evidence of clinical readiness.

In [10]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

def metric_row(name, truth, pred):
    return {
        "model": name,
        "accuracy": accuracy_score(truth, pred),
        "precision": precision_score(truth, pred, zero_division=0),
        "recall": recall_score(truth, pred, zero_division=0),
        "f1": f1_score(truth, pred, zero_division=0),
    }

metric_rows = [
    metric_row("Classical ML (TF-IDF + Logistic Regression)", y_test, ml_pred),
    metric_row("Ensemble (rule + ML; documented fallback)", y_test, majority_predictions),
]
covered = rule_test.notna()
if covered.any():
    metric_rows.append(metric_row("Rule-based (covered cases only)", y_test[covered], rule_test[covered].astype(int)))

metrics_df = pd.DataFrame(metric_rows)
display(metrics_df.round(3))
print("Classical ML confusion matrix [rows=true, columns=predicted]:")
print(confusion_matrix(y_test, ml_pred))
print(classification_report(y_test, ml_pred, zero_division=0))

,model,accuracy,precision,recall,f1
0,Classical ML (TF-IDF + Logistic Regression),0.983,1.0,0.964,0.982
1,Ensemble (rule + ML; documented fallback),0.983,1.0,0.964,0.982
2,Rule-based (covered cases only),1.000,1.0,1.000,1.000


Classical ML confusion matrix [rows=true, columns=predicted]:
[[64  0]
 [ 2 54]]
              precision    recall  f1-score   support

           0       0.97      1.00      0.98        64
           1       1.00      0.96      0.98        56

    accuracy                           0.98       120
   macro avg       0.98      0.98      0.98       120
weighted avg       0.98      0.98      0.98       120



## 11. Inspect cases needing review

In [11]:
flagged = results[results["human_review"]]
print(f"Flagged for review: {len(flagged)} of {len(results)} test reports")
display(flagged[["report", "actual_lvi_label", "rule_prediction", "ml_prediction",
                 "ml_score_lvi_present", "review_reason"]].head(20))

Flagged for review: 107 of 120 test reports


,report,actual_lvi_label,rule_prediction,ml_prediction,ml_score_lvi_present,review_reason
0,Neural involvement: None.\nSentinel lymph node...,0,NaN,0,0.233401,Rule-based model uncertain/conflicting
1,LVI noted. Sentinel lymph node biopsy negative...,1,NaN,1,0.764762,Rule-based model uncertain/conflicting
2,Neural involvement: None.\nDiagnosis: Invasive...,1,NaN,1,0.575152,Rule-based model uncertain/conflicting; ML sco...
3,Evidence of lymphovascular invasion. No DCIS i...,1,NaN,1,0.544390,Rule-based model uncertain/conflicting; ML sco...
4,"Diagnosis: Invasive ductal carcinoma, Grade II...",0,NaN,0,0.345160,Rule-based model uncertain/conflicting
5,LVI not seen. Perineural invasion: absent. His...,0,NaN,0,0.180351,Rule-based model uncertain/conflicting
6,Margins involved by tumor.\nPerineural invasio...,1,NaN,1,0.773590,Rule-based model uncertain/conflicting
7,Evidence of lymphovascular invasion. HER2: equ...,1,NaN,1,0.532677,Rule-based model uncertain/conflicting; ML sco...
9,Extensive necrosis present.\nNo necrosis obser...,1,NaN,0,0.495434,Rule-based model uncertain/conflicting; ML sco...
10,no lymphovascular or perineural invasion obser...,0,NaN,0,0.152320,Rule-based model uncertain/conflicting


## 12. Export test predictions

In [12]:
OUTPUT_PATH = "/content/ensemble_lvi_test_results.csv"
results.to_csv(OUTPUT_PATH, index=False)
print("Saved:", OUTPUT_PATH)
from google.colab import files
files.download(OUTPUT_PATH)

Saved: /content/ensemble_lvi_test_results.csv


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

## 13. Key takeaways

- Models can fail differently: terminology variation, context, negation, and unsupported outputs.
- Ensembling is not guaranteed to improve accuracy; the method and diversity of errors matter.
- Majority voting, probability averaging, and weighted confidence are different methods.
- A model score is not automatically a calibrated probability.
- Flag disagreement and uncertainty for human review; clinical thresholds require expert validation.
- This starter notebook implements a rule-based baseline and classical ML. Transformer training is optional; real LLM integration is not implemented until configured.
- Nottingham grade needs its own labelled target column and evaluation. The screenshot shows `lvi_label`, so this notebook focuses on LVI.